SOAL 1 — Data Loading & Profiling Awal

In [17]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField, StringType, DateType, TimestampType

In [18]:
spark = SparkSession.builder \
    .appName("ecommerce_profiling") \
    .master("local[*]") \
    .getOrCreate()

BASE = "dataset/"


In [13]:
spark.sparkContext.setLogLevel("ERROR")

In [19]:
orders_schema = StructType([
    StructField("order_id",     StringType()),
    StructField("user_id",      StringType()),
    StructField("order_date",   DateType()),
    StructField("status",       StringType()),
    StructField("total_amount", DoubleType()),
])
order_items_schema = StructType([
    StructField("order_item_id", StringType()),
    StructField("order_id",      StringType()),
    StructField("product_id",    StringType()),
    StructField("quantity",      IntegerType()),
    StructField("price",         DoubleType()),
])
products_schema = StructType([
    StructField("product_id",   StringType()),
    StructField("product_name", StringType()),
    StructField("category",     StringType()),
    StructField("price",        DoubleType()),
])
users_schema = StructType([
    StructField("user_id",     StringType()),
    StructField("name",        StringType()),
    StructField("email",       StringType()),
    StructField("signup_date", DateType()),
])

In [21]:
def load(name, schema):
    return (spark.read.option("header", True)
            .option("dateFormat", "yyyy-MM-dd")
            .option("mode", "PERMISSIVE")
            .schema(schema)
            .csv(f"{BASE}{name}.csv"))

In [22]:

orders      = load("orders", orders_schema)
order_items = load("order_items", order_items_schema)
products    = load("products", products_schema)
users       = load("users", users_schema)

dfs = {"orders": orders, "order_items": order_items,
       "products": products, "users": users}

In [ ]:
print("=== Jumlah baris tiap DataFrame ===")
for table, row in dfs.items():
    print(f"{table}: {row.count()}")


print("\n=== cek null per kolom ===")
for table, row in dfs.items():
    print(f"\n[{table}]")
    row.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                for c in row.columns]).show(truncate=False)

print("\n=== Sample 5 baris ===")
for table, row in dfs.items():
    print(f"\n[{table}]")
    row.show(5, truncate=False)


=== Row count ===
orders: 18
order_items: 20
products: 10
users: 14

=== Null per kolom ===

[orders]
+--------+-------+----------+------+------------+
|order_id|user_id|order_date|status|total_amount|
+--------+-------+----------+------+------------+
|1       |1      |0         |0     |18          |
+--------+-------+----------+------+------------+


[order_items]
+-------------+--------+----------+--------+-----+
|order_item_id|order_id|product_id|quantity|price|
+-------------+--------+----------+--------+-----+
|1            |1       |0         |20      |20   |
+-------------+--------+----------+--------+-----+


[products]
+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|1         |0           |0       |0    |
+----------+------------+--------+-----+


[users]
+-------+----+-----+-----------+
|user_id|name|email|signup_date|
+-------+----+-----+-----------+
|1      |0   |0    |0          |
+-------+----+---

In [39]:
print("=== jumlah orders dengan total_amount <= 0 ===")

orders_bad_amount = orders.filter(F.col("total_amount") <= 0).count()
print(f"orders: total_amount <= 0: {orders_bad_amount}")

print("\n=== jumlah order_items dengan quantity <= 0 atau price <= 0 ===")
items_bad_qty_or_price = order_items.filter(
    (F.col("quantity") <= 0) | (F.col("price") <= 0)
).count()

print(f"order_items: quantity <= 0 or price <= 0: {items_bad_qty_or_price}")

print("\n=== jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users) ===")
def count_duplicates(df, key):
    return df.groupBy(key).count().filter(F.col("count") > 1).count()

print(f"orders: duplicate order_id: {count_duplicates(orders, 'order_id')}")
print(f"products: duplicate product_id: {count_duplicates(products, 'product_id')}")
print(f"users: duplicate user_id: {count_duplicates(users, 'user_id')}")




=== jumlah orders dengan total_amount <= 0 ===
orders: total_amount <= 0: 0

=== jumlah order_items dengan quantity <= 0 atau price <= 0 ===
order_items: quantity <= 0 or price <= 0: 0

=== jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users) ===
orders: duplicate order_id: 1
products: duplicate product_id: 1
users: duplicate user_id: 1


SOAL 2 — Cleaning & Standardization
Perintah 1:
Bersihkan data:
drop baris orders yang order_id atau user_id null,
drop baris order_items yang order_id atau product_id null,
drop baris products yang product_id null,
drop baris users yang user_id null.

Perintah 2:
Standardisasi kolom teks:
city di users → lowercase + trim,
category di products → lowercase + trim.
Perintah 3:
Deduplication berbasis key:
orders dedupe by order_id,
order_items dedupe by kombinasi (order_id,product_id),
products dedupe by product_id,
users dedupe by user_id.
 Tunjukkan jumlah baris sebelum vs sesudah dedupe.


In [ ]:
# ================ Perintah 1: Bersihkan data ======================

print("=== drop baris orders yang order_id atau user_id null ===")
df.
